In [7]:
import pandas as pd
from pandas.errors import EmptyDataError

def safe_read_csv(path, required_columns):
    try:
        df = pd.read_csv(path)
        if df.empty:
            raise EmptyDataError
        return df
    except (FileNotFoundError, EmptyDataError):
        print(f"⚠️ Warning: {path} missing or empty. Using empty fallback.")
        return pd.DataFrame(columns=required_columns)


In [8]:
import cv2
import pandas as pd
import numpy as np
import os

# ================= CONFIG =================
VIDEO_PATH = r"C:\Users\Ayush\Desktop\Coding World\Multi_Modal_SnatchDetection\Dataset\Video\Snatch_Shoot\new\Shoot_8.mp4"  # CHANGE THIS
SAVE_VIDEO = True
OUTPUT_VIDEO_PATH = r"C:\Users\Ayush\Desktop\Coding World\Multi_Modal_SnatchDetection\ffffff.mp4"

# Colors (BGR)
COLOR_PERSON = (0, 255, 0)
COLOR_OBJECT = (0, 165, 255)
COLOR_VELOCITY = (255, 255, 0)
COLOR_APPROACH = (0, 0, 255)
COLOR_SAFE = (255, 0, 0)
COLOR_SNATCH_HAND = (0, 255, 255)
COLOR_HANDSHAKE = (255, 0, 255)
COLOR_TEXT = (255, 255, 255)

FONT = cv2.FONT_HERSHEY_SIMPLEX


In [9]:
print("Loading CSVs...")

df_entities = pd.read_csv("frame_entities.csv")
df_motion   = pd.read_csv("person_motion.csv")

df_pair = safe_read_csv(
    "person_pair_interaction.csv",
    [
        "frame_id","person_i_id","person_j_id",
        "distance_ground","distance_delta",
        "relative_speed","motion_asymmetry_score",
        "approach_velocity","bearing_score",
        "approach_flag","separation_flag","parallel_flag"
    ]
)

df_hand = safe_read_csv(
    "hand_interaction.csv",
    [
        "frame_id","person_i_id","person_j_id","hand_side",
        "hand_center_x","hand_center_y",
        "distance_hand_to_person_j","distance_hand_delta",
        "contact_duration_frames","contact_zone_ratio"
    ]
)

df_object = safe_read_csv(
    "object_interaction.csv",
    [
        "frame_id","object_id","class_name","person_id",
        "distance_object_to_person","distance_delta",
        "present_flag","disappearance_flag",
        "attacker_velocity_match"
    ]
)

def index_by_frame(df):
    if df.empty:
        return {}
    return {k: v for k, v in df.groupby("frame_id")}

entities_map = index_by_frame(df_entities)
motion_map   = index_by_frame(df_motion)
pair_map     = index_by_frame(df_pair)
hand_map     = index_by_frame(df_hand)
object_map   = index_by_frame(df_object)

print("✅ Data loaded & indexed (SAFE MODE).")


Loading CSVs...
✅ Data loaded & indexed (SAFE MODE).


In [10]:
import os
VIDEO_ID = os.path.splitext(os.path.basename(VIDEO_PATH))[0]

try:
    df_features = pd.read_csv(f"{VIDEO_ID}_features.csv")
    print(f"✅ Loaded features for {VIDEO_ID}")
except FileNotFoundError:
    print(f"⚠️ Warning: {VIDEO_ID}_features.csv not found.")
    df_features = pd.DataFrame(columns=["start_frame", "end_frame", "label"])


⚠️ Warning: Shoot_8_features.csv not found.


In [11]:
def draw_text(img, text, pos, color=COLOR_TEXT, scale=0.5):
    x, y = pos
    cv2.putText(img, text, (x, y), FONT, scale, (0,0,0), 3)
    cv2.putText(img, text, (x, y), FONT, scale, color, 1)

def draw_arrow(img, start, vec, scale=5, color=COLOR_VELOCITY):
    end = (int(start[0] + vec[0]*scale), int(start[1] + vec[1]*scale))
    cv2.arrowedLine(img, start, end, color, 2, tipLength=0.3)

def get_person_center(pid, frame_entities):
    rows = frame_entities[
        (frame_entities["entity_type"] == "person") &
        (frame_entities["entity_id"] == pid)
    ]
    if len(rows) > 0:
        r = rows.iloc[0]
        return int(r["bbox_center_x"]), int(r["bbox_center_y"])
    return None


In [12]:
cap = cv2.VideoCapture(VIDEO_PATH)
fps = int(cap.get(cv2.CAP_PROP_FPS))

if SAVE_VIDEO:
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(
        OUTPUT_VIDEO_PATH,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w, h)
    )

frame_id = 0
paused = False

print("Controls: SPACE = Pause | N = Next | Q = Quit")

while cap.isOpened():
    if not paused:
        ret, frame = cap.read()
        if not ret:
            break
        frame_id += 1
        base_frame = frame.copy()
    else:
        frame = base_frame.copy()

    ents = entities_map.get(frame_id, pd.DataFrame())
    mots = motion_map.get(frame_id, pd.DataFrame())
    pairs = pair_map.get(frame_id, pd.DataFrame())
    hands = hand_map.get(frame_id, pd.DataFrame())
    objs  = object_map.get(frame_id, pd.DataFrame())

    # ---------- ENTITIES ----------
    for _, e in ents.iterrows():
        x1,y1,x2,y2 = map(int,[e.x1,e.y1,e.x2,e.y2])
        cx,cy = int(e.bbox_center_x), int(e.bbox_center_y)
        eid = int(e.entity_id)

        if e.entity_type == "person":
            cv2.rectangle(frame,(x1,y1),(x2,y2),COLOR_PERSON,2)
            draw_text(frame,f"P{eid}",(x1,y1-8),COLOR_PERSON)

            m = mots[mots.person_id == eid]
            if not m.empty:
                vx,vy = m.iloc[0].velocity_x, m.iloc[0].velocity_y
                if abs(vx)>2 or abs(vy)>2:
                    draw_arrow(frame,(cx,cy),(vx,vy))

        else:
            cv2.rectangle(frame,(x1,y1),(x2,y2),COLOR_OBJECT,2)
            draw_text(frame,f"{e.class_name}",(x1,y1-8),COLOR_OBJECT)

    # ---------- PAIR INTERACTIONS ----------
    for _, p in pairs.iterrows():
        c1 = get_person_center(int(p.person_i_id), ents)
        c2 = get_person_center(int(p.person_j_id), ents)
        if c1 and c2:
            if p.approach_velocity < -2:
                cv2.line(frame,c1,c2,COLOR_APPROACH,3)
                draw_text(frame,f"Approach {p.approach_velocity:.1f}",
                          ((c1[0]+c2[0])//2,(c1[1]+c2[1])//2),COLOR_APPROACH)
            else:
                cv2.line(frame,c1,c2,COLOR_SAFE,1)

    # ---------- HAND CONTACT ----------
    for _, h in hands.iterrows():
        hx,hy = int(h.hand_center_x), int(h.hand_center_y)
        dur = int(h.contact_duration_frames)
        color = COLOR_SNATCH_HAND if dur < 10 else COLOR_HANDSHAKE
        cv2.circle(frame,(hx,hy),7,color,-1)
        draw_text(frame,f"{dur}",(hx+8,hy),color)

    # ---------- OBJECT EVENTS ----------
    for _, o in objs.iterrows():
        if o.disappearance_flag == 1:
            draw_text(frame,f"🚨 LOST {o.class_name}",(50,100),(0,0,255),1)

    # ---------- FINAL EVENT ALERT (TABLE 6) ----------
    for _, w in df_features.iterrows():
        if w["start_frame"] <= frame_id <= w["end_frame"] and w["label"] == 1:
            draw_text(
                frame,
                "🚨 LABELED SNATCH (Ground Truth)",
                (50, 60),
                (0, 0, 255),
                1.0
            )
            break


    draw_text(frame,f"Frame {frame_id}",(20,30))

    cv2.imshow("Validation Debugger",frame)
    if SAVE_VIDEO:
        out.write(frame)

    key = cv2.waitKey(30)&0xFF
    if key==ord('q'): break
    elif key==ord(' '): paused=not paused
    elif key==ord('n') and paused: paused=False

cap.release()
if SAVE_VIDEO: out.release()
cv2.destroyAllWindows()


Controls: SPACE = Pause | N = Next | Q = Quit
